In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, pointbiserialr
from scipy.stats import f_oneway, kruskal

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score

import xgboost as xgb

plt.style.use('default')
sns.set_palette("husl")
SEED = 43
np.random.seed(SEED)

SCORING = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error'
}
GRID_SEARCH_KWARGS = dict(
    cv=10,
    scoring=SCORING,
    refit='r2',
    n_jobs=-1,
    return_train_score=True
)
LINEAR_PARAM_GRID = {
    'model__fit_intercept': [True, False],
    'model__positive': [False, True]
}
TREE_PARAM_GRID = {
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
RF_PARAM_GRID = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
MLP_PARAM_GRID = {
    'model__hidden_layer_sizes': [(100,), (100, 50), (50, 25)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [0.0001, 0.001, 0.01],
    'model__learning_rate_init': [0.001, 0.01]
}
XGB_PARAM_GRID = {
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__n_estimators': [50, 100, 200]
}


In [2]:
data = pd.read_csv("../data/data_yolo.csv", index_col=0)
print(data)

      country time_of_day        lat       long       road_type  \
8459       SE         day  55.604209  13.028574            city   
63960      DE         day  50.936889   6.908079  arterial-urban   
71801      IT         day  41.987014  12.496538         highway   
85589      IT       night  41.794853  12.522880         highway   
90649      DE         day  49.182153   9.414737         highway   
...       ...         ...        ...        ...             ...   
74962      PL         day  54.378439  18.605984  arterial-urban   
90673      PL         day  50.253217  19.823732   smaller-rural   
24373      DE         day  50.931399   6.953686            city   
41597      SE         day  55.608149  13.003458            city   
40353      IT       night  41.918667  12.383545            city   

      road_condition            weather  solar_angle_elevation  month  hour  \
8459          normal             cloudy              36.560248      5    14   
63960         normal               ra

In [3]:
data = data.dropna().reset_index(drop=True)
num_rows, num_cols = data.shape
print(f"After dropping missing values: {num_rows:,} rows and {num_cols} columns")

After dropping missing values: 10,006 rows and 38 columns


In [4]:
iou = data["iou"]
lrp = data["lrp"]
conf = data["conf"]

data = data.drop(columns=["iou", "lrp"])
data_indices = data.index.to_numpy()

In [5]:
all_columns = data.columns    
all_columns = all_columns.drop(["weather_code", "cloud_cover_low", "cloud_cover_mid"])

print(f"Number of rows: {data.shape[0]}")
print(f"Columns: {all_columns.sort_values()}")
print(f"Number of columns: {len(all_columns)}")

Number of rows: 10006
Columns: Index(['brightness', 'camera_distance_from_ground', 'camera_offset',
       'camera_pitch_angle', 'cloud_cover', 'complexity', 'conf', 'contrast',
       'country', 'distortion_magnitude', 'field_view_horizontal',
       'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat',
       'lateral_acceleration', 'lateral_velocity', 'long', 'month',
       'noisiness', 'quality', 'rain', 'relative_humidity_2m',
       'road_condition', 'road_type', 'sharpness', 'snowfall',
       'solar_angle_elevation', 'temperature_2m', 'time_of_day', 'weather',
       'wind_speed_10m'],
      dtype='object')
Number of columns: 33


In [6]:
numeric_columns = ['conf', 'solar_angle_elevation', 'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall', 'cloud_cover', 'wind_speed_10m', 'forward_acceleration', 'lateral_acceleration', 'forward_velocity', 'lateral_velocity', 'field_view_horizontal', 'camera_distance_from_ground', 'camera_pitch_angle', 'distortion_magnitude', 'camera_offset', 'laplacian', 'brightness', 'contrast', 'sharpness', 'noisiness', 'quality', 'complexity']

categorical_columns = ["country", "time_of_day", "road_type", "road_condition", "weather"]
other = ["hour", "month", "lat", "long"]
print(sorted(numeric_columns + categorical_columns + other))
assert len(all_columns) == (len(categorical_columns) + len(numeric_columns) + len(other)), "Columns not match"

['brightness', 'camera_distance_from_ground', 'camera_offset', 'camera_pitch_angle', 'cloud_cover', 'complexity', 'conf', 'contrast', 'country', 'distortion_magnitude', 'field_view_horizontal', 'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat', 'lateral_acceleration', 'lateral_velocity', 'long', 'month', 'noisiness', 'quality', 'rain', 'relative_humidity_2m', 'road_condition', 'road_type', 'sharpness', 'snowfall', 'solar_angle_elevation', 'temperature_2m', 'time_of_day', 'weather', 'wind_speed_10m']


# Assessor Tests

Split data into train 60% and validation 40%

In [7]:
#X_train, X_test, y_train, y_test = train_test_split(data, test_size=0.4, train_size=0.6, stratify=data["iou"])
train_idx, test_idx = train_test_split(data_indices, test_size=0.4, train_size=0.6)

X_train = data.loc[train_idx]
y_train = iou.loc[train_idx]
y_train_lrp = lrp.loc[train_idx]
conf_train = conf.loc[train_idx]
conf_train = pd.DataFrame(conf_train)

X_test = data.loc[test_idx]
y_test = iou.loc[test_idx]
y_test_lrp = lrp.loc[test_idx]
conf_test = conf.loc[test_idx]
conf_test = pd.DataFrame(conf_test)

print("X:", len(X_train), len(X_test))
print("y:", len(y_train), len(y_test))
print(X_train.columns)

X: 6003 4003
y: 6003 4003
Index(['country', 'time_of_day', 'lat', 'long', 'road_type', 'road_condition',
       'weather', 'solar_angle_elevation', 'month', 'hour',
       'forward_acceleration', 'lateral_acceleration', 'forward_velocity',
       'lateral_velocity', 'field_view_horizontal',
       'camera_distance_from_ground', 'camera_pitch_angle',
       'distortion_magnitude', 'camera_offset', 'laplacian', 'quality',
       'brightness', 'noisiness', 'sharpness', 'contrast', 'complexity',
       'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall',
       'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'wind_speed_10m',
       'weather_code', 'conf'],
      dtype='object')


In [8]:
model_results = []

### IoU

#### Baseline

Random Prediction. Mean of metric over training set.

In [9]:
mean_iou_train = np.mean(y_train)
iou_pred_test = np.full_like(y_test, mean_iou_train)
mean_baseline_r2 = r2_score(y_test, iou_pred_test)
mean_baseline_mae = np.mean(np.abs(y_test - iou_pred_test))
mean_baseline_mse = np.mean((y_test - iou_pred_test)**2)
print(f"R2 score of mean baseline: {mean_baseline_r2:.4f}")
print(f"MAE of mean baseline: {mean_baseline_mae:.4f}")
print(f"MSE of mean baseline: {mean_baseline_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Constant Mean Predictor',
    'test_r2': mean_baseline_r2,
    'test_mae': mean_baseline_mae,
    'test_mse': mean_baseline_mse,
})


R2 score of mean baseline: -0.0004
MAE of mean baseline: 0.1227
MSE of mean baseline: 0.0273


#### Linear Reg on Conf

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_conf_grid = GridSearchCV(
    linear_reg_conf_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_grid.fit(conf_train, y_train)

best_idx = linear_reg_conf_grid.best_index_
mean_train_r2 = linear_reg_conf_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_conf_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_conf_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_conf_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_conf_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_conf_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_conf_grid.best_params_}")
best_linear_reg_conf = linear_reg_conf_grid.best_estimator_


Mean CV train r2 score 0.3408
Mean CV test r2 score 0.3394
Mean CV train MAE score 0.1014
Mean CV test MAE score 0.1015
Mean CV train MSE score 0.0181
Mean CV test MSE score 0.0181
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [11]:
y_pred = best_linear_reg_conf.predict(conf_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Univariate Linear Regression (Confidence)',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3493
MAE test score 0.0996
MSE test score 0.0177


#### Linear Regression

Train models with cv and then test.

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_grid = GridSearchCV(
    linear_reg_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_grid.fit(X_train, y_train)

best_idx = linear_reg_grid.best_index_
mean_train_r2 = linear_reg_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_grid.best_params_}")
best_linear_reg = linear_reg_grid.best_estimator_


Mean CV train r2 score 0.3753
Mean CV test r2 score 0.3556
Mean CV train MAE score 0.0992
Mean CV test MAE score 0.1005
Mean CV train MSE score 0.0172
Mean CV test MSE score 0.0177
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [13]:
y_pred = best_linear_reg.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Linear Regression',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3679
MAE test score 0.0984
MSE test score 0.0172


#### Decision Trees

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_grid = GridSearchCV(
    decision_tree_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_grid.fit(X_train, y_train)

best_idx = decision_tree_grid.best_index_
mean_train_r2 = decision_tree_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_grid.best_params_}")
best_decision_tree = decision_tree_grid.best_estimator_


/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/base.py", line 1363, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda

Mean CV train r2 score 0.4001
Mean CV test r2 score 0.3323
Mean CV train MAE score 0.0962
Mean CV test MAE score 0.1015
Mean CV train MSE score 0.0165
Mean CV test MSE score 0.0184
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


In [15]:
y_pred = best_decision_tree.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Decision Tree',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3425
MAE test score 0.0997
MSE test score 0.0179


#### Random Forest

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_grid = GridSearchCV(
    rand_forest_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_grid.fit(X_train, y_train)

best_idx = rand_forest_grid.best_index_
mean_train_r2 = rand_forest_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_grid.best_params_}")
best_rand_forest = rand_forest_grid.best_estimator_


/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/base.py", line 1363, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda

Mean CV train r2 score 0.4843
Mean CV test r2 score 0.3627
Mean CV train MAE score 0.0900
Mean CV test MAE score 0.0990
Mean CV train MSE score 0.0142
Mean CV test MSE score 0.0175
Best params: {'model__max_depth': 10, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 200}


In [17]:
y_pred = best_rand_forest.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Random Forest',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3772
MAE test score 0.0969
MSE test score 0.0170


#### MLP

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_grid = GridSearchCV(
    mlp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_grid.fit(X_train, y_train)

best_idx = mlp_grid.best_index_
mean_train_r2 = mlp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_grid.best_params_}")
best_mlp = mlp_grid.best_estimator_


Mean CV train r2 score 0.3790
Mean CV test r2 score 0.2738
Mean CV train MAE score 0.0999
Mean CV test MAE score 0.1069
Mean CV train MSE score 0.0171
Mean CV test MSE score 0.0199
Best params: {'model__activation': 'tanh', 'model__alpha': 0.01, 'model__hidden_layer_sizes': (50, 25), 'model__learning_rate_init': 0.001}


In [19]:
y_pred = best_mlp.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'MLP',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3279
MAE test score 0.1023
MSE test score 0.0183


#### XGBoost

In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_grid.fit(X_train, y_train)


best_idx = xgb_grid.best_index_
mean_train_r2 = xgb_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_grid.best_params_}")
best_xgb = xgb_grid.best_estimator_

Mean CV train r2 score 0.5142
Mean CV test r2 score 0.3464
Mean CV train MAE score 0.0898
Mean CV test MAE score 0.1014
Mean CV train MSE score 0.0134
Mean CV test MSE score 0.0179
Best params: {'model__learning_rate': 0.01, 'model__n_estimators': 200}


In [21]:
y_pred = best_xgb.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'XGBoost',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3618
MAE test score 0.0997
MSE test score 0.0174


### LRP


#### Baseline

Predict Only the Mean


In [22]:
mean_lrp_train = np.mean(y_train_lrp)
lrp_pred_test = np.full_like(y_test_lrp, mean_lrp_train)
mean_lrp_r2 = r2_score(y_test_lrp, lrp_pred_test)
mean_lrp_mae = np.mean(np.abs(y_test_lrp - lrp_pred_test))
mean_lrp_mse = np.mean((y_test_lrp - lrp_pred_test)**2)
print(f"R2 score of random baseline: {mean_lrp_r2:.4f}")
print(f"MAE of random baseline: {mean_lrp_mae:.4f}")
print(f"MSE of random baseline: {mean_lrp_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Constant Mean Predictor',
    'test_r2': mean_lrp_r2,
    'test_mae': mean_lrp_mae,
    'test_mse': mean_lrp_mse,
})


R2 score of random baseline: -0.0002
MAE of random baseline: 0.1054
MSE of random baseline: 0.0206


#### Linear Reg on Conf

In [23]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_conf_lrp_grid = GridSearchCV(
    linear_reg_conf_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_lrp_grid.fit(conf_train, y_train_lrp)

best_idx = linear_reg_conf_lrp_grid.best_index_
mean_train_r2 = linear_reg_conf_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_conf_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_conf_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_conf_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_conf_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_conf_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_conf_lrp_grid.best_params_}")
best_linear_reg_conf_lrp = linear_reg_conf_lrp_grid.best_estimator_


Mean CV train r2 score 0.3990
Mean CV test r2 score 0.3975
Mean CV train MAE score 0.0827
Mean CV test MAE score 0.0827
Mean CV train MSE score 0.0126
Mean CV test MSE score 0.0127
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [24]:
y_pred = best_linear_reg_conf_lrp.predict(conf_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Univariate Linear Regression (Confidence)',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.3946
MAE test score 0.0822
MSE test score 0.0125


#### Linear Regression


Train models with cv and then test.


In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_lrp_grid = GridSearchCV(
    linear_reg_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_lrp_grid.fit(X_train, y_train_lrp)

best_idx = linear_reg_lrp_grid.best_index_
mean_train_r2 = linear_reg_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_lrp_grid.best_params_}")
best_linear_reg_lrp = linear_reg_lrp_grid.best_estimator_


Mean CV train r2 score 0.4405
Mean CV test r2 score 0.4199
Mean CV train MAE score 0.0796
Mean CV test MAE score 0.0808
Mean CV train MSE score 0.0118
Mean CV test MSE score 0.0122
Best params: {'model__fit_intercept': False, 'model__positive': False}


In [26]:
y_pred = best_linear_reg_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Linear Regression',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4208
MAE test score 0.0802
MSE test score 0.0119


#### Decision Trees


In [27]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_lrp_grid = GridSearchCV(
    decision_tree_lrp_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_lrp_grid.fit(X_train, y_train_lrp)

best_idx = decision_tree_lrp_grid.best_index_
mean_train_r2 = decision_tree_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_lrp_grid.best_params_}")
best_decision_tree_lrp = decision_tree_lrp_grid.best_estimator_


/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/base.py", line 1363, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda

Mean CV train r2 score 0.4632
Mean CV test r2 score 0.4041
Mean CV train MAE score 0.0764
Mean CV test MAE score 0.0802
Mean CV train MSE score 0.0113
Mean CV test MSE score 0.0125
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


In [28]:
y_pred = best_decision_tree_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Decision Tree',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4108
MAE test score 0.0785
MSE test score 0.0121


#### Random Forest


In [29]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_lrp_grid = GridSearchCV(
    rand_forest_lrp_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_lrp_grid.fit(X_train, y_train_lrp)

best_idx = rand_forest_lrp_grid.best_index_
mean_train_r2 = rand_forest_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_lrp_grid.best_params_}")
best_rand_forest_lrp = rand_forest_lrp_grid.best_estimator_


/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/base.py", line 1363, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda

Mean CV train r2 score 0.5521
Mean CV test r2 score 0.4354
Mean CV train MAE score 0.0704
Mean CV test MAE score 0.0780
Mean CV train MSE score 0.0094
Mean CV test MSE score 0.0118
Best params: {'model__max_depth': 10, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 200}


In [30]:
y_pred = best_rand_forest_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Random Forest',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4329
MAE test score 0.0772
MSE test score 0.0117


#### MLP


In [31]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_lrp_grid = GridSearchCV(
    mlp_lrp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_lrp_grid.fit(X_train, y_train_lrp)

best_idx = mlp_lrp_grid.best_index_
mean_train_r2 = mlp_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_lrp_grid.best_params_}")
best_mlp_lrp = mlp_lrp_grid.best_estimator_


Mean CV train r2 score 0.3630
Mean CV test r2 score 0.3370
Mean CV train MAE score 0.0857
Mean CV test MAE score 0.0870
Mean CV train MSE score 0.0134
Mean CV test MSE score 0.0139
Best params: {'model__activation': 'relu', 'model__alpha': 0.001, 'model__hidden_layer_sizes': (50, 25), 'model__learning_rate_init': 0.01}


In [32]:
y_pred = best_mlp_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'MLP',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4052
MAE test score 0.0805
MSE test score 0.0123


#### XGBoost

In [33]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_lrp_grid = GridSearchCV(
    xgb_lrp_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_lrp_grid.fit(X_train, y_train_lrp)


best_idx = xgb_lrp_grid.best_index_
mean_train_r2 = xgb_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_lrp_grid.best_params_}")
best_xgb_lrp = xgb_lrp_grid.best_estimator_

Mean CV train r2 score 0.6028
Mean CV test r2 score 0.4192
Mean CV train MAE score 0.0693
Mean CV test MAE score 0.0797
Mean CV train MSE score 0.0084
Mean CV test MSE score 0.0122
Best params: {'model__learning_rate': 0.01, 'model__n_estimators': 200}


In [34]:
y_pred = best_xgb_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'XGBoost',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4130
MAE test score 0.0794
MSE test score 0.0121


### Final Model Comparison


In [35]:
results_df = pd.DataFrame(model_results, index=None)
results_df["test_r2"] = results_df["test_r2"].round(4)
results_df["test_mae"] = results_df["test_mae"].round(4)
results_df["test_mse"] = results_df["test_mse"].round(4)
results_df.to_csv("results.csv")
display(results_df)


,target,model,test_r2,test_mae,test_mse
0,IoU,Constant Mean Predictor,-0.0004,0.1227,0.0273
1,IoU,Univariate Linear Regression (Confidence),0.3493,0.0996,0.0177
2,IoU,Linear Regression,0.3679,0.0984,0.0172
3,IoU,Decision Tree,0.3425,0.0997,0.0179
4,IoU,Random Forest,0.3772,0.0969,0.0170
5,IoU,MLP,0.3279,0.1023,0.0183
6,IoU,XGBoost,0.3618,0.0997,0.0174
7,LRP,Constant Mean Predictor,-0.0002,0.1054,0.0206
8,LRP,Univariate Linear Regression (Confidence),0.3946,0.0822,0.0125
9,LRP,Linear Regression,0.4208,0.0802,0.0119
